# Elliptic curves from scratch
How does elliptic-curve cryptography compute with points, and what do ECDH and ECDSA do with them?
We compute with the EC key from `02` ourselves and cross-check with OpenSSL.

Run `02_openssl_keys.ipynb` first (it creates the key files). Run cells from top to bottom. Modify the cells marked `TODO` (the check cells after them verify your implementation).

Written answers (**Your answer** cells) are not checked automatically – after you finish a task, verify them against the notebook in `solutions/`.

The ASN.1/DER support code is provided (an EC grammar for keys and signatures, and `der_values`, a variant of `der_to_values` from `03`, to read the curve parameters) –
the tasks are only about the EC mathematics.

In [ ]:
import os

for name in ["EC_256_priv.der", "EC_256_priv.pem", "EC_256_pub.pem"]:
    assert os.path.exists(name), f"{name} is missing - run 02_openssl_keys.ipynb first"

Run this cell once. The EC key from `02` stores only the name of its curve (`prime256v1`); `P256_params.der` contains the actual values $p, a, b, G, n$
needed for our calculations. The cell also creates a key pair for Bob and lets OpenSSL compute the ECDH shared secret from both sides.

In [ ]:
!openssl ecparam -name prime256v1 -param_enc explicit -outform DER -out P256_params.der
!openssl genpkey -algorithm EC -outform PEM -pkeyopt ec_paramgen_curve:P-256 -pkeyopt ec_param_enc:named_curve -out EC_B_priv.pem
!openssl pkey -inform PEM -outform DER -in EC_B_priv.pem -out EC_B_priv.der
!openssl pkey -pubout -inform PEM -outform PEM -in EC_B_priv.pem -out EC_B_pub.pem
!openssl pkeyutl -derive -inkey EC_256_priv.pem -peerkey EC_B_pub.pem -out ECDH_AB.bin
!openssl pkeyutl -derive -inkey EC_B_priv.pem -peerkey EC_256_pub.pem -out ECDH_BA.bin
open("PT.txt", "w").write("Hello PV181");

In [ ]:
import hashlib
import secrets

import asn1
import asn1tools

def der_values(path):
    """All primitive values of a DER file, in order (like der_to_values in 03)."""
    decoder = asn1.Decoder()
    decoder.start(open(path, "rb").read())
    values = []
    def walk():
        while not decoder.eof():
            tag = decoder.peek()
            if tag is None:  # end of a nested structure
                return
            if tag.typ == asn1.Types.Primitive:
                values.append(decoder.read()[1])
            else:
                decoder.enter()
                walk()
                decoder.leave()
    walk()
    return values

def point(data):
    """Uncompressed point 04 || x || y -> (x, y)."""
    assert data[0] == 0x04 and len(data) == 65
    return int.from_bytes(data[1:33], "big"), int.from_bytes(data[33:], "big")

# EC grammar: private key (RFC 5915) and ECDSA signature (RFC 5480)
ec_spec = asn1tools.compile_string("""
EC DEFINITIONS ::= BEGIN
    ECPrivateKey ::= SEQUENCE {
        version        INTEGER,
        privateKey     OCTET STRING,
        parameters [0] EXPLICIT OBJECT IDENTIFIER OPTIONAL,
        publicKey  [1] EXPLICIT BIT STRING OPTIONAL
    }
    ECDSA-Sig-Value ::= SEQUENCE {
        r INTEGER,
        s INTEGER
    }
END
""")

## Part 1: How point arithmetic works
The curve P-256 is $y^2 = x^3 + ax + b \pmod p$ with a generator point $G$ of order $n$; the parameters are read from `P256_params.der`.
Points are added with the chord-and-tangent rule; `O` (the point at infinity) is the neutral element. For $P = (x_1, y_1)$, $Q = (x_2, y_2)$:
slope $s = \frac{y_2 - y_1}{x_2 - x_1}$ if $P \ne Q$, $s = \frac{3x_1^2 + a}{2y_1}$ if $P = Q$ (division = multiplication by the inverse mod $p$), then
$x_3 = s^2 - x_1 - x_2$, $y_3 = s(x_1 - x_3) - y_1 \pmod p$.

$k \cdot P = P + P + \dots + P$ is computed by **double-and-add** – square-and-multiply from `05` written with points: double instead of square, add instead of multiply.
`mul` returns a pair `(point, number_of_additions)`.

In [ ]:
params = der_values("P256_params.der")  # values in order: version, field OID, p, a, b, seed, G, n, cofactor h
p = params[2]
a = int.from_bytes(params[3], "big")
b = int.from_bytes(params[4], "big")
G = point(params[6])
n = params[7]

O = None  # point at infinity

def on_curve(P):
    x, y = P
    return (y * y - (x * x * x + a * x + b)) % p == 0

def add(P, Q):
    if P is O:
        return Q
    if Q is O:
        return P
    x1, y1 = P
    x2, y2 = Q
    if x1 == x2 and (y1 + y2) % p == 0:  # P + (-P) = O
        return O
    if P == Q:
        s = (3 * x1 * x1 + a) * pow(2 * y1, -1, p) % p   # tangent
    else:
        s = (y2 - y1) * pow(x2 - x1, -1, p) % p          # chord
    x3 = (s * s - x1 - x2) % p
    return (x3, (s * (x1 - x3) - y1) % p)

def mul_naive(k, P):
    R, steps = O, 0
    for _ in range(k):
        R = add(R, P)
        steps += 1
    return R, steps

def mul(k, P):
    """k*P (k >= 1) and the number of point additions."""
    bits = bin(k)[2:]
    R, steps = P, 0            # the leading bit is 1
    for bit in bits[1:]:
        R = add(R, R)          # double
        steps += 1
        if bit == "1":
            R = add(R, P)      # add
            steps += 1
    return R, steps

assert on_curve(G) and on_curve(add(G, G))
R1, s1 = mul_naive(10_000, G)
R2, s2 = mul(10_000, G)
assert R1 == R2
print(f"k = 10000: naive {s1} additions, double-and-add {s2}")
print(f"256-bit k: double-and-add needs {mul(secrets.randbelow(n), G)[1]} additions, the naive way about 2^256")
assert mul(n, G)[0] is O  # G has order n

## Part 2: EC with the key from `02`
An EC private key is a number $d$, the public key is the point $Q = d \cdot G$.

In [ ]:
key = ec_spec.decode("ECPrivateKey", open("EC_256_priv.der", "rb").read())
d = int.from_bytes(key["privateKey"], "big")
Q = point(key["publicKey"][0])

bob = ec_spec.decode("ECPrivateKey", open("EC_B_priv.der", "rb").read())
dB = int.from_bytes(bob["privateKey"], "big")
QB = point(bob["publicKey"][0])

### Task 1: Public key from the private key
Compute the public key of the OpenSSL key with your `mul`; the check compares it with the public key stored in the file.

In [ ]:
Q_computed = ...  # TODO: d * G with mul (it returns (point, steps) - use [0])

In [ ]:
assert Q_computed == Q, "Q_computed is not the public key stored in the file"
print("Q = d * G holds for the OpenSSL key")

**a)** Why can't the private key $d$ be computed from the public key $Q$?

**Your answer** (double-click to edit)

**a)** …

### Task 2: ECDH
You have $(d, Q)$, Bob has $(d_B, Q_B)$. Each side combines its own private key with the other's public key.
OpenSSL returns only the x-coordinate of the shared point. Compute the shared point and compare with OpenSSL.
(Here we only compare the raw x-coordinate; a real protocol passes it through a key-derivation function before using it as a symmetric key.)

In [ ]:
S = ...  # TODO: your side of ECDH

In [ ]:
assert S == mul(dB, Q)[0], "S is not the point computed on Bob's side"
secret = S[0].to_bytes(32, "big")
print("own    :", secret.hex())
print("OpenSSL:", open("ECDH_AB.bin", "rb").read().hex())
assert secret == open("ECDH_AB.bin", "rb").read() == open("ECDH_BA.bin", "rb").read()

**a)** Why do both sides get the same shared point?

**Your answer** (double-click to edit)

**a)** …

### Task 3: You sign, OpenSSL verifies
ECDSA signature of a message with hash $h$ (SHA-256 as an integer) and a random secret $k$:
$r = (k \cdot G)_x \bmod n$, $\quad s = k^{-1} (h + r \cdot d) \bmod n$.
($r$ and $s$ must not be 0 – a new $k$ would be chosen then; for random 256-bit $k$ this practically never happens.)
You implement signing; the signature is encoded as DER `SEQUENCE { r, s }`, and OpenSSL performs the verification with the public key from `02`.

In [ ]:
h = int.from_bytes(hashlib.sha256(open("PT.txt", "rb").read()).digest(), "big")
k = secrets.randbelow(n - 1) + 1
r = ...  # TODO
s = ...  # TODO

In [ ]:
open("my_sig.der", "wb").write(ec_spec.encode("ECDSA-Sig-Value", {"r": r, "s": s}))
!openssl asn1parse -i -inform DER -in my_sig.der
check = !openssl dgst -sha256 -verify EC_256_pub.pem -signature my_sig.der PT.txt
print(*check, sep="\n")
assert any("Verified OK" in line for line in check)

**a)** Which values are needed to verify the signature, and which must stay secret?

**Your answer** (double-click to edit)

**a)** …

### Optional: why $k$ must never repeat
$k$ must be secret, unpredictable and never reused – knowing it or reusing it reveals the private key. If two messages are signed with the same $k$, both signatures have the same $r$, and anybody can compute
$k = (h_1 - h_2)/(s_1 - s_2) \bmod n$ and then the private key $d = (s_1 k - h_1)/r \bmod n$. (This is how the Sony PS3 signing key was recovered.)

In [ ]:
k = secrets.randbelow(n - 1) + 1  # the same k for both messages - a mistake
r = mul(k, G)[0][0] % n
h1 = int.from_bytes(hashlib.sha256(b"message 1").digest(), "big")
h2 = int.from_bytes(hashlib.sha256(b"message 2").digest(), "big")
s1 = pow(k, -1, n) * (h1 + r * d) % n
s2 = pow(k, -1, n) * (h2 + r * d) % n

k_found = (h1 - h2) * pow(s1 - s2, -1, n) % n
d_found = (s1 * k_found - h1) * pow(r, -1, n) % n
print("private key recovered:", d_found == d)